In [1]:
from IPython.display import clear_output
import warnings
warnings.filterwarnings('ignore')
clear_output(wait=True)

In [2]:
clear_output(wait=True)
print('hello world')

hello world


In [3]:
import time 
for i in range(5):
    clear_output(wait=True)
    print(f"Step {i+1} completed")
    time.sleep(1)

Step 5 completed


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

from IPython.display import display, clear_output
import pandas as pd 

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)


print('librarires imported successfully!!')


librarires imported successfully!!


In [5]:
# stop spark session is there any 
try : 
    spark.stop()
    print('stopped previous spark session!!')

except : 
    pass 

# start new session 

try: 
    from pyspark import SparkContext
    sc = SparkContext._active_spark_context
    if sc : 
        sc.stop()
        print('stopped previous sparkContext !!')
except : 
    pass 


spark = SparkSession.builder\
    .appName('02-Single aggregation using the local machine')\
    .master('local[*]')\
    .config('spark.sql.adaptive.enabled', 'true')\
    .config('spark.sql.debug.maxToStringFields', '1000')\
    .getOrCreate()

spark.conf.set('spark.sql.debug.maxToStringFields', '1000')

print('Spark started successfully !!')
print(f'Spark version : {spark.version}')

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/25 14:09:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/25 14:09:49 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark started successfully !!
Spark version : 3.5.1


### Step 2-a : Load Fire Departement Data 

In [6]:
# Define the schema 

fire_schema = StructType([StructField('CallNumber', IntegerType(), True),
StructField('UnitID', StringType(), True),
StructField('IncidentNumber', IntegerType(), True),
StructField('CallType', StringType(), True),                  
StructField('CallDate', StringType(), True),      
StructField('WatchDate', StringType(), True),
StructField('CallFinalDisposition', StringType(), True),
StructField('AvailableDtTm', StringType(), True),
StructField('Address', StringType(), True),       
StructField('City', StringType(), True),       
StructField('Zipcode', IntegerType(), True),       
StructField('Battalion', StringType(), True),                 
StructField('StationArea', StringType(), True),       
StructField('Box', StringType(), True),       
StructField('OriginalPriority', StringType(), True),       
StructField('Priority', StringType(), True),       
StructField('FinalPriority', IntegerType(), True),       
StructField('ALSUnit', BooleanType(), True),       
StructField('CallTypeGroup', StringType(), True),
StructField('NumAlarms', IntegerType(), True),
StructField('UnitType', StringType(), True),
StructField('UnitSequenceInCallDispatch', IntegerType(), True),
StructField('FirePreventionDistrict', StringType(), True),
StructField('SupervisorDistrict', StringType(), True),
StructField('Neighborhood', StringType(), True),
StructField('Location', StringType(), True),
StructField('RowID', StringType(), True),
StructField('Delay', FloatType(), True)])

# Set up the data source file's path 
sf_fire_file = '/opt/spark-data/sf-fire-calls.csv'

# load dataframe with San Francesco fire data 

fire_df = spark.read.csv(sf_fire_file, header= True, schema = fire_schema)
        #.option('samplingRatio', 0.00001)\ # to infere schema

print(' --------- data is loaded successfully --------- ')
print(f'Number of rows is : {fire_df.count()}')
print(f'Number of columns is {len(fire_df.columns)}')

 --------- data is loaded successfully --------- 


[Stage 0:===============>                                          (3 + 8) / 11]

Number of rows is : 175296
Number of columns is 28


### Step 2 : Prepare Data For Parquet Storage 
Before aggrefatioins, let's prepare the data : 
1. clean column names (remove spaces from partition values)
2. write as partitioned parquet 
3. Compare CSV vs Parquet performance and metadata 

In [7]:
# Create clean partition columns (no spaces, no special caracters)
# Also extract data for data-based partitionning 
# Replace problematic caracters for file system compatibility : 
#   - spaces -> underscores
#   - forward slash -> underscore (can cause directory issues)
#   - parentheses -> removed (can cause shell issues)

fire_df_clean = fire_df.withColumn(
        'CallType_Clean', 
        regexp_replace(
            regexp_replace(
                regexp_replace(
                    col('CallType'), r"[\s/]", "_"
                ), 
                r"[()]", ""
            ),
            r"_+", "_"
        )
    )\
    .withColumn(
        'Neighborhood_Clean',
        regexp_replace(
            regexp_replace(
                col('Neighborhood'),
                r"[\s/]", "_"
            ),
            r"_+", "_"            
        )
    )\
    .withColumn(
        'call_date',
        to_date(col('CallDate'), 'MM/dd/yyyy')
    )


print('created clean partition columns')
print('Original VS Clean column examples :')

result = fire_df_clean.select(
    'CallDate', 'call_date',
    'CallType', 'CallType_Clean',
    'Neighborhood', 'Neighborhood_Clean'

).distinct()

pdf = result.limit(20).toPandas()

display(pdf)


created clean partition columns
Original VS Clean column examples :


,CallDate,call_date,CallType,CallType_Clean,Neighborhood,Neighborhood_Clean
0,01/12/2002,2002-01-12,Medical Incident,Medical_Incident,Twin Peaks,Twin_Peaks
1,01/19/2002,2002-01-19,Structure Fire,Structure_Fire,Nob Hill,Nob_Hill
2,01/26/2002,2002-01-26,Medical Incident,Medical_Incident,Oceanview/Merced/Ingleside,Oceanview_Merced_Ingleside
3,01/26/2002,2002-01-26,Citizen Assist / Service Call,Citizen_Assist_Service_Call,Inner Sunset,Inner_Sunset
4,02/09/2002,2002-02-09,Medical Incident,Medical_Incident,South of Market,South_of_Market
5,02/20/2002,2002-02-20,Structure Fire,Structure_Fire,West of Twin Peaks,West_of_Twin_Peaks
6,02/27/2002,2002-02-27,Medical Incident,Medical_Incident,Oceanview/Merced/Ingleside,Oceanview_Merced_Ingleside
7,03/11/2002,2002-03-11,Alarms,Alarms,Pacific Heights,Pacific_Heights
8,03/13/2002,2002-03-13,Alarms,Alarms,South of Market,South_of_Market
9,03/27/2002,2002-03-27,Medical Incident,Medical_Incident,Outer Mission,Outer_Mission


### Step 2-b : Write Partitioned Parquet Files 
Parquet advantages : 
- Columnar storage - faster for analytics
- Compression - smaller file size 
- Schema embedded - no need to define schema when reading 
- Partitionning - faster question on partition columns 

In [ ]:
print('Writing multilevel partioned parquet file ...')
print('Hierarchy : Date -> CallType -> Priority')
print()

fire_df_clean.write.mode('overwrite')\
    .partitionBy('call_date', 'CallType_Clean', 'Priority')\
    .parquet('/opt/spark-data/output2/fire-aggregations-partitionned')

print('parquet filres written to : /opt/spark-data/output/fire-aggregations-partitionned/')

Writing multilevel partioned parquet file ...
Hierarchy : Date -> CallType -> Priority



26/09/25 14:11:21 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/09/25 14:11:21 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/09/25 14:11:21 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/09/25 14:11:21 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/09/25 14:11:22 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/09/25 14:11:22 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/09/25 14:11:22 WARN MemoryManager: Total allocation exceeds 95.0

In [ ]:
#fire_df_clean.select('call_date').agg(min(),max())
spark.stop()